# Cross-object action-state review

This manual review uses wrist RGB from three support recordings. A shared observation tracks the same piece, the gripper's hold, and the piece's relation to a free support or a specific assembly site. Aperture is a separate cue. The reference labels select examples for inspection. Before-window and after-window frames are retrospective context. Frames inside the window can also fall between the model's saved samples.

In [ ]:
import json
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np

repo = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "main.py").is_file())
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from src.video_processing import read_sampled_frames

root = repo / "outputs/visual_reassemble_hand"
plan = json.loads((root / "plan.json").read_text(encoding="utf-8"))
cases = {
    "2025-01-09-13-59-54_hand_clip_0001": [25, 45, 74, 104, 134, 153],
    "2025-01-09-13-59-54_hand_clip_0088": [3941, 3960, 3989, 4019, 4049, 4067],
    "2025-01-09-13-59-54_hand_clip_0092": [4121, 4140, 4169, 4199, 4229, 4249],
    "2025-01-09-13-59-54_hand_clip_0048": [2025, 2090, 2140, 2249, 2270],
    "2025-01-09-15-02-21_hand_clip_0138": [6080, 6150, 6220, 6260, 6299, 6320],
    "2025-01-09-15-02-21_hand_clip_0042": [1880, 1910, 1950, 1960, 1979, 2000],
    "2025-01-09-14-28-47_hand_clip_0395": [17757, 17775, 17804, 17834, 17864, 17884],
}

for clip_id, indices in cases.items():
    video_id = clip_id.rsplit("_clip_", 1)[0]
    directory = root / video_id
    clips = json.loads((directory / "clips.json").read_text(encoding="utf-8"))
    clip, = [row for row in clips if row["clip_id"] == clip_id]
    assert clip["recording_id"] in plan["split"]["support"]
    assert clip["camera"] == "hand"
    timestamps = np.load(directory / f"{video_id}_timestamps.npy", allow_pickle=False).reshape(-1)
    assert np.array_equal(timestamps[clip["sampled_frame_indices"]], clip["original_frame_times"])
    frames = read_sampled_frames(directory / f"{video_id}.mp4", indices)
    fig, axes = plt.subplots(1, len(indices), figsize=(16, 3), layout="constrained")
    fig.suptitle(f"{clip_id} | segment {clip['segment_id']} | "
                 f"{clip['ground_truth']} {clip['object'] or ''} | success={clip['success']}")
    for ax, index, frame in zip(axes, indices, frames, strict=True):
        if index in clip["sampled_frame_indices"]:
            scope = "Model sample"
        elif clip["start_frame"] <= index < clip["end_frame_exclusive"]:
            scope = "In window, between samples"
        else:
            scope = "Context outside window"
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{scope}\nFrame {index} | {timestamps[index]!s}", fontsize=8)
        ax.axis("off")
    plt.show()
    plt.close(fig)

## Manual observations

| Clip suffix | Initial observation | Interaction and later evidence |
|---|---|---|
| 0001 | Open jaws, no piece visible between them | The view shifts across the board while the jaws stay empty. The reviewed frames show no grasp or release, consistent with the IDLE annotation. |
| 0088 | Small gear on a free surface and empty jaws | The jaws approach, then the gear is visible between the jaws at the last sampled frame 4049. This supports PICK from a free surface. |
| 0092 | Small gear held between jaws | The jaws open by the last sample 4229. In context frame 4249 the gear is away from the gripper and appears to rest on the free surface. |
| 0048 (USB) | Plug held and aligned with the board receptacle | The joint is partly hidden while held. In context frame 2270 the jaws are open and the plug remains at the receptacle. This supports seating after release. |
| 0138 (BNC) | Piece at its fixed socket before grasp in context frame 6080 | The jaws hold the BNC during the window. In context frame 6320 the socket and the BNC in the jaws are visibly separate. |
| 0042 (round peg 4) | Peg at its board hole before grasp in context frame 1880 | The peg is gripped around the window end. More of the shank and the rim of the hole are visible at frame 2000. Full separation is hard to confirm from this view. |
| 0395 (bolt 4) | Threaded piece held near a board site | The gripper keeps hold through the reviewed frames. Engagement and the outcome are visually uncertain. The dataset separately annotates a failed INSERT attempt. |

BNC and the round peg use the same evidence criterion for REMOVE. The later BNC image shows a gap, while the sampled peg sequence leaves full separation uncertain. The underlying evidence can differ by visibility. Keep the assembly site type as a property of the scene and preserve unknown values when a relation is hidden.